# Shakespeare Transformer on MLX

This notebook trains a roughly **4.8M-parameter** character Transformer on the local `data/shakespeare.txt` corpus. MLX Visualizer opens in your browser and updates model weights, loss, validation loss, and throughput while the training cell runs.

Run the cells in order. The **Start training** cell is the only long-running cell.

In [ ]:
# Locate the repository and install MLX only if this kernel does not have it.
from importlib.util import find_spec
from pathlib import Path
import subprocess
import sys

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src/mlx_visualizer").is_dir()
)
if find_spec("mlx") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlx>=0.5"])
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))
print(f"Repository: {REPO_ROOT}")

In [ ]:
import mlx.core as mx
import mlx.optimizers as optim
from IPython.display import HTML, display

from mlx_visualizer import Visualizer
from examples.shakespeare_transformer import (
    ShakespeareTransformer, TrainingState, TransformerConfig,
    find_shakespeare_path, generate_text, load_corpus, parameter_count,
    save_checkpoint, train_model,
)

DATA_PATH = find_shakespeare_path(REPO_ROOT / "data/shakespeare.txt")
corpus = load_corpus(DATA_PATH)
print(f"Corpus: {DATA_PATH}")
print(f"Characters: {len(corpus.train) + len(corpus.validation):,} | vocabulary: {corpus.vocab_size}")

In [ ]:
# The defaults are intentionally well below the requested 10M-parameter ceiling.
CONFIG = TransformerConfig(
    context_length=128, model_dim=256, num_heads=8, num_layers=6, mlp_dim=1024
)
mx.random.seed(7)
model = ShakespeareTransformer(corpus.vocab_size, CONFIG)
mx.eval(model.parameters())
params = parameter_count(model)
assert params < 10_000_000
print(f"Parameters: {params:,} ({params / 1e6:.2f}M)")
model

## Start the live visualizer

This opens the viewer in a browser tab. Leave it open, then run the training cell below. Grid mode mixes live metric charts with parameter heatmaps; Architecture mode shows the discovered model graph.

In [ ]:
state = TrainingState()
viz = Visualizer(port=0, interval=0.25, max_side=256, tick_budget=0.05)
viz.watch_module(
    "transformer", model,
    sample_input=mx.zeros((1, CONFIG.context_length), dtype=mx.int32),
    every=4,
    param_filter=lambda _path, key: key == "weight",
    staged=True,  # MLX GPU arrays are copied on this notebook thread
)
viz.metric("training/loss", lambda: state.loss, group="training", history=500)
viz.metric("training/validation_loss", lambda: state.validation_loss, group="training", history=500)
viz.metric("training/tokens_per_second", lambda: state.tokens_per_second, group="training", history=500, colormap="viridis")
url = viz.start(open_browser=True)
display(HTML(f'<a href="{url}" target="_blank"><b>Open MLX Visualizer</b></a>'))

## Start training

`STEPS=1500` is a useful first run. Increase it after confirming everything works; you can interrupt the cell at any time and still sample or save the current model. Every report calls `viz.refresh()` on this training thread, so the background viewer never evaluates MLX GPU arrays directly.

In [ ]:
STEPS = 1500
BATCH_SIZE = 32
optimizer = optim.AdamW(learning_rate=3e-4, weight_decay=1e-2)
state = train_model(
    model, optimizer, corpus,
    steps=STEPS, batch_size=BATCH_SIZE,
    report_every=10, evaluate_every=100, evaluation_batches=10,
    state=state,
    callback=lambda _state: viz.refresh(),
)

In [ ]:
sample = generate_text(
    model, corpus, "ROMEO:", length=800, temperature=0.8, seed=12
)
print(sample)

In [ ]:
weights_path = save_checkpoint(model, corpus, REPO_ROOT / "artifacts")
print(f"Saved: {weights_path}")
# Run viz.stop() when you are finished with the live viewer.